In [108]:
import pandas as pd
import os
import numpy as np
import re
from tqdm import tqdm

### Original HVU Files

In [2]:
hvu_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/HVU_Train_V1.0.csv")
hvu_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/HVU_Val_V1.0.csv")
#HVU Labels dataset
hvu_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/HVU_Tags_Categories_V1.0.csv")

In [3]:
hvu_train_df.head()

,Tags,youtube_id,time_start,time_end
0,brass_instrument|musical_instrument|trumpet|wo...,OLpWTpTC4P8,57.0,67.0
1,gadget|electronic_musical_instrument|electroni...,pjj6zedpCQY,75.0,85.0
2,photograph|joint|shoulder|nature|human_leg|fun...,_STSOsITd0E,3.0,13.0
3,mouth|child|fun|nose|hair|human_hair_color|hea...,0r_BGL3416g,406.0,416.0
4,photograph|violist|cellist|violin_family|night...,xsPKW4tZZBc,233.0,243.0


In [4]:
hvu_val_df.head()

,Tags,youtube_id,time_start,time_end
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0


In [9]:
print(f"Total items in HVU Train file: {hvu_train_df.shape[0]}")
print(f"Total items in HVU Val file: {hvu_val_df.shape[0]}")
print(f"Total items in HVU Label file: {hvu_labels['Tag'].nunique()}")

Total items in HVU Train file: 481417
Total items in HVU Val file: 31171
Total items in HVU Label file: 3142


In [115]:
hvu_labels.head()

,Tag,Category
0,playing_trombone,action
1,playing_controller,action
2,photograph,action
3,recreation,action
4,sitting,action


In [20]:
hvu_train_files = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/train_file.csv")
hvu_val_files = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/val_file.csv")
hvu_test_files = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/test_file.csv")

In [21]:
# Train set
train_tags = hvu_train_df['Tags'].str.split("|").explode().str.strip()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Training set: {unique_train_count}")

# Val Set
val_tags = hvu_val_df['Tags'].str.split("|").explode().str.strip()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Validation set: {unique_val_count}")

Total Unique Tags in Training set: 3142
Total Unique Tags in Validation set: 3142


In [22]:
def extract_timestamps_from_stripped_filename(filename):
    original_filename = filename
    # Use regex to split by underscore while ignoring empty segments caused by consecutive underscores
    parts = [part for part in re.split(r'(.*?)(\d+\.?\d*)_(\d+\.?\d*)\.mp4', filename) if part]
    # print(parts)
    
    # Assuming the last two parts of the filename are the timestamps
    if len(parts) >= 2:  # Ensure there are at least two parts
        start_time = float(parts[-2])
        end_time = float(parts[-1])
        filename = parts[0][:-1]
        return original_filename, filename, start_time, end_time
    
    return filename, None, None

In [23]:
# get youtube_id, time_start and time_end from the filename
hvu_val_files[['original_filename','filename', 'time_start', 'time_end']] = hvu_val_files['filename'].apply(lambda x: pd.Series(extract_timestamps_from_stripped_filename(x)))
hvu_train_files[['original_filename','filename', 'time_start', 'time_end']] = hvu_train_files['filename'].apply(lambda x: pd.Series(extract_timestamps_from_stripped_filename(x)))

In [24]:
hvu_val_files = hvu_val_files.rename(columns = {'filename': 'youtube_id'})
hvu_train_files = hvu_train_files.rename(columns = {'filename': 'youtube_id'})

In [25]:
# Combine the val_df with the val_files (filenames)
hvu_val_combined = hvu_val_df.merge(hvu_val_files, left_on=['youtube_id', 'time_start', 'time_end'], right_on=['youtube_id', 'time_start', 'time_end'], suffixes=('_df', '_file'), how='left')
hvu_train_combined = hvu_train_df.merge(hvu_train_files, left_on=['youtube_id', 'time_start', 'time_end'], right_on=['youtube_id', 'time_start', 'time_end'], suffixes=('_df', '_file'), how='left')

In [26]:
hvu_val_combined.head()

,Tags,youtube_id,time_start,time_end,original_filename
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5,gD_G1b0wV5I_101.5_103.5.mp4
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5,eeEC-oWWAM0_90.5_92.5.mp4
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5,Np0jqmKPvSQ_170.5_172.5.mp4
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0,s2I4-ErSc7E_182.0_184.0.mp4
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0,TLZXKaS2abQ_31.0_33.0.mp4


In [27]:
hvu_train_combined.head()

,Tags,youtube_id,time_start,time_end,original_filename
0,brass_instrument|musical_instrument|trumpet|wo...,OLpWTpTC4P8,57.0,67.0,OLpWTpTC4P8_000057_000067.mp4
1,gadget|electronic_musical_instrument|electroni...,pjj6zedpCQY,75.0,85.0,pjj6zedpCQY_000075_000085.mp4
2,photograph|joint|shoulder|nature|human_leg|fun...,_STSOsITd0E,3.0,13.0,_STSOsITd0E_000003_000013.mp4
3,mouth|child|fun|nose|hair|human_hair_color|hea...,0r_BGL3416g,406.0,416.0,0r_BGL3416g_000406_000416.mp4
4,photograph|violist|cellist|violin_family|night...,xsPKW4tZZBc,233.0,243.0,xsPKW4tZZBc_000233_000243.mp4


In [28]:
print(f"Total items in HVU train dataset after combining: {hvu_train_combined.shape[0]}")
print(f"Total items in HVU val dataset after combining: {hvu_val_combined.shape[0]}")

Total items in HVU train dataset after combining: 481417
Total items in HVU val dataset after combining: 31171


### Individual File Statistics

In [30]:
full_action_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/action_labels.csv", names=['tag', 'label'])
full_attribute_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/attribute_labels.csv", names=['tag', 'label'])
full_concept_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/concept_labels.csv", names=['tag', 'label'])
full_event_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/event_labels.csv", names=['tag', 'label'])
full_object_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/object_labels.csv", names=['tag', 'label'])
full_scene_labels = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/label_files/scene_labels.csv", names=['tag', 'label'])


print(f"Total items in Action Label file: {full_action_labels['tag'].nunique()}")
print(f"Total items in Attribute Label file: {full_attribute_labels['tag'].nunique()}")
print(f"Total items in Concept Label file: {full_concept_labels['tag'].nunique()}")
print(f"Total items in Event Label file: {full_event_labels['tag'].nunique()}")
print(f"Total items in Object Label file: {full_object_labels['tag'].nunique()}")
print(f"Total items in Scene Label file: {full_scene_labels['tag'].nunique()}")




Total items in Action Label file: 739
Total items in Attribute Label file: 117
Total items in Concept Label file: 291
Total items in Event Label file: 69
Total items in Object Label file: 1678
Total items in Scene Label file: 248


In [35]:
full_action_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/action/train.csv", names = ['filename', 'labels'])
full_action_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/action/val.csv", names = ['filename', 'labels'])

full_attribute_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/attribute/train.csv", names = ['filename', 'labels'])
full_attribute_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/attribute/val.csv", names = ['filename', 'labels'])

full_concept_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/concept/train.csv", names = ['filename', 'labels'])
full_concept_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/concept/val.csv", names = ['filename', 'labels'])

full_event_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/event/train.csv", names = ['filename', 'labels'])
full_event_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/event/val.csv", names = ['filename', 'labels'])

full_object_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/object/train.csv", names = ['filename', 'labels'])
full_object_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/object/val.csv", names = ['filename', 'labels'])

full_scene_train_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/scene/train.csv", names = ['filename', 'labels'])
full_scene_val_df = pd.read_csv("/home/ahmad/cvr/HVU-Dataset/multilabel_files/scene/val.csv", names = ['filename', 'labels'])


In [80]:
# Train set
train_tags = full_action_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_action_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Action Training set: {unique_train_count}")
print(f"Total Unique Videos in Action Training  set: {count_train_files}")

print()
val_tags = full_action_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_action_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Action Val set: {unique_val_count}")
print(f"Total Unique Videos in Action Val  set: {count_val_files}")

Total Unique Tags in Action Training set: 739
Total Unique Videos in Action Training  set: 479568

Total Unique Tags in Action Val set: 739
Total Unique Videos in Action Val  set: 31082


In [82]:
# Train set
train_tags = full_attribute_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_attribute_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Attribute Training set: {unique_train_count}")
print(f"Total Unique Videos in Attribute Training  set: {count_train_files}")

print()

val_tags = full_attribute_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_attribute_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Attribute Val set: {unique_val_count}")
print(f"Total Unique Videos in Attribute Val  set: {count_val_files}")


Total Unique Tags in Action Training set: 117
Total Unique Videos in Attribute Training  set: 316040

Total Unique Tags in Attribute Val set: 117
Total Unique Videos in Attribute Val  set: 23387


In [84]:
# Train set
train_tags = full_concept_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_concept_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Action Training set: {unique_train_count}")
print(f"Total Unique Videos in Concept Training  set: {count_train_files}")

print()
val_tags = full_concept_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_concept_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Concept Val set: {unique_val_count}")
print(f"Total Unique Videos in Concept Val  set: {count_val_files}")


Total Unique Tags in Action Training set: 291
Total Unique Videos in Concept Training  set: 410711

Total Unique Tags in Concept Val set: 291
Total Unique Videos in Concept Val  set: 27946


In [88]:
# Train set
train_tags = full_event_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_event_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Event Training set: {unique_train_count}")
print(f"Total Unique Videos in Event Training  set: {count_train_files}")

print()

val_tags = full_event_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_event_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Event Val set: {unique_val_count}")
print(f"Total Unique Videos in Event Val  set: {count_val_files}")


Total Unique Tags in Event Training set: 69
Total Unique Videos in Event Training  set: 164924

Total Unique Tags in Event Val set: 69
Total Unique Videos in Event Val  set: 10864


In [92]:
# Train set
train_tags = full_object_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_object_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Object Training set: {unique_train_count}")
print(f"Total Unique Videos in Object Training  set: {count_train_files}")

print()

val_tags = full_object_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_object_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Object Val set: {unique_val_count}")
print(f"Total Unique Videos in Object Val  set: {count_val_files}")


Total Unique Tags in Object Training set: 1678
Total Unique Videos in Object Training  set: 471068

Total Unique Tags in Object Val set: 1678
Total Unique Videos in Object Val  set: 30898


In [94]:
# Train set
train_tags = full_scene_train_df['labels'].str.split("|").explode().str.strip()
count_train_files = full_scene_train_df['filename'].nunique()
unique_train_tags = train_tags.unique()
unique_train_count = len(unique_train_tags)
print(f"Total Unique Tags in Scene Training set: {unique_train_count}")
print(f"Total Unique Videos in Scene Training  set: {count_train_files}")

print()

val_tags = full_scene_val_df['labels'].str.split("|").explode().str.strip()
count_val_files = full_scene_val_df['filename'].nunique()
unique_val_tags = val_tags.unique()
unique_val_count = len(unique_val_tags)
print(f"Total Unique Tags in Scene Val set: {unique_val_count}")
print(f"Total Unique Videos in Scene Val  set: {count_val_files}")


Total Unique Tags in Scene Training set: 248
Total Unique Videos in Scene Training  set: 251794

Total Unique Tags in Scene Val set: 248
Total Unique Videos in Scene Val  set: 16200


### CVDF K400

In [48]:
cvd_train_df = pd.read_csv("/home/ahmad/cvr/zero_shot/cvdf_datasets/k400/train.csv")
cvd_val_df = pd.read_csv("/home/ahmad/cvr/zero_shot/cvdf_datasets/k400/val.csv")
cvd_test_df = pd.read_csv("/home/ahmad/cvr/zero_shot/cvdf_datasets/k400/test.csv")

In [49]:
cvd_train_df['filename'] = cvd_train_df['youtube_id'] + '_' + cvd_train_df['time_start'].astype(str).str.zfill(6) + '_' + cvd_train_df['time_end'].astype(str).str.zfill(6) + '.mp4'
cvd_val_df['filename'] = cvd_val_df['youtube_id'] + '_' + cvd_val_df['time_start'].astype(str).str.zfill(6) + '_' + cvd_val_df['time_end'].astype(str).str.zfill(6) + '.mp4'
cvd_test_df['filename'] = cvd_test_df['youtube_id'] + '_' + cvd_test_df['time_start'].astype(str).str.zfill(6) + '_' + cvd_test_df['time_end'].astype(str).str.zfill(6) + '.mp4'

In [50]:
cvd_train_df.head()

,label,youtube_id,time_start,time_end,split,is_cc,filename
0,abseiling,-3B32lodo2M,59,69,train,0,-3B32lodo2M_000059_000069.mp4
1,abseiling,-7kbO0v4hag,107,117,train,0,-7kbO0v4hag_000107_000117.mp4
2,abseiling,-bwYZwnwb8E,13,23,train,0,-bwYZwnwb8E_000013_000023.mp4
3,abseiling,-Cv3NwxG_8g,87,97,train,0,-Cv3NwxG_8g_000087_000097.mp4
4,abseiling,-hLv_HL6UhY,151,161,train,0,-hLv_HL6UhY_000151_000161.mp4


In [51]:
# Add split_name to identify which part the data belongs to 
cvd_train_df['split'] = 'train'
cvd_val_df['split'] = 'val'
cvd_test_df['split'] = 'test'

In [52]:
print(f"Total items in train file: {cvd_train_df.shape[0]}")
print(f"Total items in val file: {cvd_val_df.shape[0]}")
print(f"Total items in test file: {cvd_test_df.shape[0]}")

Total items in train file: 246534
Total items in val file: 19906
Total items in test file: 39805


In [53]:
# Unique labels in dataset
print(f"Unique labels in Train set: {cvd_train_df['label'].nunique()}")
print(f"Unique labels in Val set: {cvd_val_df['label'].nunique()}")
print(f"Unique labels in Test set: {cvd_test_df['label'].nunique()}")

Unique labels in Train set: 400
Unique labels in Val set: 400
Unique labels in Test set: 400


#### Label List

In [54]:
cvd_label_list = pd.read_csv("/home/ahmad/cvr/zero_shot/K400/cvdf/kinetics_400_labels.csv")
cvd_label_list.head()

,id,name
0,0,abseiling
1,1,air drumming
2,2,answering questions
3,3,applauding
4,4,applying cream


In [55]:
def clean_label(label):
    try:
        if label:
            # Replace parentheses , convert to lowercase, and normalize spaces
            cleaned = label.replace('(', '').replace(')', '').strip().lower()
            # Replace multiple spaces with a single space
            cleaned = re.sub(r'\s+', ' ', cleaned)
            cleaned = cleaned.replace(' ', '_')
            return cleaned
        else:
            return None
    except Exception as e:
        print(label, e)

In [56]:
cvd_label_list['name'] = cvd_label_list['name'].apply(clean_label)
cvd_label_list.head()

,id,name
0,0,abseiling
1,1,air_drumming
2,2,answering_questions
3,3,applauding
4,4,applying_cream


#### Combined Dataset

In [57]:
cvd_combined_df = pd.concat([cvd_train_df, cvd_val_df, cvd_test_df])
print(f"Total items in Combined dataset : {cvd_combined_df.shape[0]}")

# Remove duplicates in combined dataset
cvd_combined_df = cvd_combined_df.drop_duplicates(subset='filename')
print(f"Total items after dropping duplicate items from the dataset: {cvd_combined_df.shape[0]}")

Total items in Combined dataset : 306245
Total items after dropping duplicate items from the dataset: 306245


In [58]:
# converting to float because HVU timestamps are in float
cvd_combined_df['time_start'] = cvd_combined_df['time_start'].astype(np.float32)
cvd_combined_df['time_end'] = cvd_combined_df['time_end'].astype(np.float32)

In [59]:
# Common items in HVU Val and CVD Combined
common_df = pd.merge(hvu_val_combined, cvd_combined_df, on=['youtube_id', 'time_start', 'time_end'], how='inner', suffixes=('_hvu_val', '_k400'))
print(f"Total common items between HVU Val and CVD Combined dataset: {common_df.shape[0]}")
common_df.head()

Total common items between HVU Val and CVD Combined dataset: 13800


,Tags,youtube_id,time_start,time_end,original_filename,label,split,is_cc,filename
0,finger|black_hair|fun|laughing|windshield|vehi...,Y3srcQZOhks,25.0,35.0,Y3srcQZOhks_000025_000035.mp4,laughing,val,0,Y3srcQZOhks_000025_000035.mp4
1,service|faucet|tableware|washing_dishes|glass|...,8IqHQwVqg8g,76.0,86.0,8IqHQwVqg8g_000076_000086.mp4,washing dishes,train,0,8IqHQwVqg8g_000076_000086.mp4
2,electronics|art|performance|guitarist|electron...,ap1IV_BKAXo,4.0,14.0,ap1IV_BKAXo_000004_000014.mp4,playing bass guitar,val,0,ap1IV_BKAXo_000004_000014.mp4
3,furniture|sitting|clothing|trousers|room|servi...,IhiKd5JSrgo,337.0,347.0,IhiKd5JSrgo_000337_000347.mp4,washing feet,val,0,IhiKd5JSrgo_000337_000347.mp4
4,plant|leisure|fun|male|nature|tree|grass|recre...,DMDe-I5VzQA,98.0,108.0,DMDe-I5VzQA_000098_000108.mp4,spinning poi,val,0,DMDe-I5VzQA_000098_000108.mp4


##### Common items between HVU and CVD based on youtube_id, time_start and time_end

#### Val File

In [60]:
# Unique items in HVU Val not in CVD Combined
in_hvu_val_df = pd.merge(hvu_val_combined, cvd_combined_df, on=['youtube_id', 'time_start', 'time_end'], how='left', suffixes=('_hvu_val', '_k400'), indicator=True)
in_hvu_val_df  = in_hvu_val_df[in_hvu_val_df['_merge'] == 'left_only']
in_hvu_val_df = in_hvu_val_df.drop(columns=['label', 'split', 'is_cc', 'filename', '_merge'])
in_hvu_val_df = in_hvu_val_df.reset_index(drop=True)
# in_hvu_val_df.drop_duplicates(subset=["youtube_id", "time_start", "time_end"])
print(f"Total items in HVU Val and not in CVD Combined dataset: {in_hvu_val_df.shape[0]}")

Total items in HVU Val and not in CVD Combined dataset: 17371


In [62]:
in_hvu_val_df.head()

,Tags,youtube_id,time_start,time_end,original_filename
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5,gD_G1b0wV5I_101.5_103.5.mp4
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5,eeEC-oWWAM0_90.5_92.5.mp4
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5,Np0jqmKPvSQ_170.5_172.5.mp4
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0,s2I4-ErSc7E_182.0_184.0.mp4
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0,TLZXKaS2abQ_31.0_33.0.mp4


In [67]:
val_tags_s1 = in_hvu_val_df['Tags'].str.split("|").explode().str.strip()
unique_val_s1_tags = val_tags_s1.unique()
unique_val_s1_count = len(unique_val_s1_tags)
print(f"Total Unique Tags in Object Val set: {unique_val_s1_count}")

Total Unique Tags in Object Val set: 3096


#### Train File

In [68]:
# Unique items in HVU Train not in CVD Combined
in_hvu_train_df = pd.merge(hvu_train_combined, cvd_combined_df, on=['youtube_id', 'time_start', 'time_end'], how='left', suffixes=('_hvu_train', '_k400'), indicator=True)
in_hvu_train_df  = in_hvu_train_df[in_hvu_train_df['_merge'] == 'left_only']
in_hvu_train_df = in_hvu_train_df.drop(columns=['label', 'split', 'is_cc', 'filename', '_merge'])
in_hvu_train_df = in_hvu_train_df.reset_index(drop=True)
print(f"Total items in HVU Train and not in CVD Combined dataset: {in_hvu_train_df.shape[0]}")

Total items in HVU Train and not in CVD Combined dataset: 310829


In [69]:
in_hvu_train_df.head()

,Tags,youtube_id,time_start,time_end,original_filename
0,musical_instrument|woodwind_instrument|flute|a...,EFTdEz8vIlA,75.0,85.0,EFTdEz8vIlA_000075_000085.mp4
1,child|loudspeaker|fun|speaker|electronic_devic...,3Qm_A6zhKSo,0.0,10.0,3Qm_A6zhKSo_000000_000010.mp4
2,skateboarding_equipment_and_supplies|photograp...,R1bhMNaMxcU,0.0,10.0,R1bhMNaMxcU_000000_000010.mp4
3,brass_instrument|musical_instrument|saxophone|...,a47xt5A2GrI,41.0,51.0,a47xt5A2GrI_000041_000051.mp4
4,musical_instrument|electronic_musical_instrume...,EdbktrBv9So,35.0,45.0,EdbktrBv9So_000035_000045.mp4


In [71]:
train_tags_s1 = in_hvu_train_df['Tags'].str.split("|").explode().str.strip()
unique_train_s1_tags = train_tags_s1.unique()
unique_train_s1_count = len(unique_train_s1_tags)
print(f"Total Unique Tags in Train set after removing videos: {unique_train_s1_count}")

Total Unique Tags in Object Val set: 3105


In [95]:
def get_categories(tags, tag_to_category):
    """Get the list of categories corresponding to the provided tags."""
    return [tag_to_category[tag] for tag in tags.split('|') if tag in tag_to_category]

In [96]:
def map_tags_to_categories(tags_file):
    """Create a dictionary mapping tags to their categories from the tags file."""
    tags_categories_df = pd.read_csv(tags_file)
    return dict(zip(tags_categories_df['Tag'], tags_categories_df['Category']))

In [97]:
def process_items_all_categories(tags, tag_to_category):
    """Process the tags to extract and join categories without filtering for main categories."""
    categories = get_categories(tags, tag_to_category)
    return '|'.join(categories)

In [100]:
tag_to_category = map_tags_to_categories("/home/ahmad/cvr/HVU-Dataset/HVU_Tags_Categories_V1.0.csv")
in_hvu_train_df['Categories'] = in_hvu_train_df['Tags'].apply(lambda tags: process_items_all_categories(tags, tag_to_category))
in_hvu_val_df['Categories'] = in_hvu_val_df['Tags'].apply(lambda tags: process_items_all_categories(tags, tag_to_category))


In [102]:
in_hvu_train_df.head()

,Tags,youtube_id,time_start,time_end,original_filename,Categories
0,musical_instrument|woodwind_instrument|flute|a...,EFTdEz8vIlA,75.0,85.0,EFTdEz8vIlA_000075_000085.mp4,object|object|object|object|object|object|acti...
1,child|loudspeaker|fun|speaker|electronic_devic...,3Qm_A6zhKSo,0.0,10.0,3Qm_A6zhKSo_000000_000010.mp4,object|object|attribute|object|object|action|o...
2,skateboarding_equipment_and_supplies|photograp...,R1bhMNaMxcU,0.0,10.0,R1bhMNaMxcU_000000_000010.mp4,object|action|object|object|attribute|object|a...
3,brass_instrument|musical_instrument|saxophone|...,a47xt5A2GrI,41.0,51.0,a47xt5A2GrI_000041_000051.mp4,object|object|object|action|object|attribute|o...
4,musical_instrument|electronic_musical_instrume...,EdbktrBv9So,35.0,45.0,EdbktrBv9So_000035_000045.mp4,object|object|object|object|attribute|object|o...


In [103]:
in_hvu_val_df.head()

,Tags,youtube_id,time_start,time_end,original_filename,Categories
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5,gD_G1b0wV5I_101.5_103.5.mp4,concept|object|event|concept|action|concept|co...
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5,eeEC-oWWAM0_90.5_92.5.mp4,scene|object|scene|scene|scene|object|object|s...
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5,Np0jqmKPvSQ_170.5_172.5.mp4,object|object|attribute|attribute|action|attri...
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0,s2I4-ErSc7E_182.0_184.0.mp4,action|object|object|attribute|object|object|o...
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0,TLZXKaS2abQ_31.0_33.0.mp4,action|attribute


In [104]:
def create_separate_dfs(dataset, main_categories, output_folder='./'):
    """Create separate CSV files for each main category in the dataset."""
    category_data = {category: [] for category in main_categories}
    
    for _, row in dataset.iterrows():
        tags = row['Tags'].split('|')
        categories = row['Categories'].split('|')
        youtube_id = row['youtube_id']
        time_start = row['time_start']
        time_end = row['time_end']
        original_filename = row['original_filename']
        
        for tag, category in zip(tags, categories):
            if category in main_categories:
                category_data[category].append([youtube_id, time_start, time_end, tag, original_filename])
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for category, data in tqdm(category_data.items()):
        if data:  # Only create files for categories that have data
            category_df = pd.DataFrame(data, columns=['youtube_id', 'time_start', 'time_end', 'tag', 'original_filename'])
            output_file_path = os.path.join(output_folder, f'{category}.csv')
            category_df.to_csv(output_file_path, index=False)

In [105]:
categories = list(hvu_labels["Category"].unique())
categories

['action', 'attribute', 'concept', 'event', 'object', 'scene']

In [109]:
create_separate_dfs(in_hvu_train_df, main_categories = categories, output_folder='./only_videos1')

100%|█████████████████████████████████████████████| 6/6 [00:15<00:00,  2.57s/it]


### Only videos label map

In [114]:
combined_tag_list = list(train_tags_s1) + list(val_tags_s1)
combined_tag_list = list(set(combined_tag_list))
print(f"Total items in combined tag list: {len(combined_tag_list)}")

Total items in combined tag list: 3109


In [117]:
filtered_hvu_label_set = hvu_labels[hvu_labels['Tag'].isin(combined_tag_list)]
filtered_hvu_label_set

,Tag,Category
0,playing_trombone,action
1,playing_controller,action
2,photograph,action
3,recreation,action
4,sitting,action
...,...,...
3137,seaside,scene
3138,riverbed,scene
3139,paddy_field,scene
3140,littoral,scene


In [118]:
for category in categories:
    cat_map = filtered_hvu_label_set[filtered_hvu_label_set['Category'] == category].copy()
    cat_map["Label"] = range(len(cat_map))
    cat_map.to_csv(f"./only_videos1/label_maps/{category}_label_map.csv", index=False)

In [122]:
for category in categories:
    category_ds = pd.read_csv(f"./only_videos1/{category}.csv")
    category_label_map = pd.read_csv(f"./only_videos1/label_maps/{category}_label_map.csv")
    merged_df = pd.merge(category_ds, category_label_map, left_on="tag", right_on="Tag", how="left")
    merged_df = merged_df.drop(columns=['Tag'])
    merged_df['Label'] = merged_df['Label'].astype(str)
    grouped_df = merged_df.groupby('original_filename').agg({
        'Label': '|'.join,
        'tag': '|'.join
    }).reset_index()
    print(len(set(list(category_ds['original_filename']))) == len(list(grouped_df['original_filename'])))
    grouped_df.to_csv(f"./only_videos1/zeroshot/{category}_zeroshot_ds.csv", index=False)

True
True
True
True
True
True


### For validation set

In [123]:
create_separate_dfs(in_hvu_val_df, main_categories = categories, output_folder='./only_videos1')

100%|█████████████████████████████████████████████| 6/6 [00:00<00:00,  6.18it/s]


In [124]:
for category in categories:
    category_ds = pd.read_csv(f"./only_videos1/{category}.csv")
    category_label_map = pd.read_csv(f"./only_videos1/label_maps/{category}_label_map.csv")
    merged_df = pd.merge(category_ds, category_label_map, left_on="tag", right_on="Tag", how="left")
    merged_df = merged_df.drop(columns=['Tag'])
    merged_df['Label'] = merged_df['Label'].astype(str)
    grouped_df = merged_df.groupby('original_filename').agg({
        'Label': '|'.join,
        'tag': '|'.join
    }).reset_index()
    print(len(set(list(category_ds['original_filename']))) == len(list(grouped_df['original_filename'])))
    grouped_df.to_csv(f"./only_videos1/zeroshot_val/{category}_zeroshot_ds.csv", index=False)

True
True
True
True
True
True
